# StreamMind — GPU Evaluation on Google Colab

This notebook runs the full StreamMind evaluation pipeline on a Colab GPU runtime and produces LaTeX-ready numbers for the paper tables.

**What it does:**
1. Installs dependencies and clones the repo
2. Verifies GPU availability
3. **Phase 1** — Latency profiling (per-component breakdown on GPU)
4. **Phase 2** — LiveQA-Bench evaluation (63 QA pairs, 3 scopes)
5. **Phase 3** — Ablation study (memory sizes, FIFO, no-TQR)
6. **Phase 4** — External benchmarks (NExT-QA, EgoSchema, OVO-Bench, Ego4D-NLQ) — *requires data upload*
7. Converts all results to LaTeX table values

**Runtime:** Select **GPU** (T4 or better) under *Runtime → Change runtime type*.

| Phase | Estimated Time (T4) | Estimated Time (A100) |
|-------|--------------------|-----------------------|
| Latency profiling | ~2 min | ~1 min |
| LiveQA-Bench (full) | ~5 min | ~2 min |
| Full ablation (6 configs) | ~30 min | ~10 min |
| NExT-QA (5,240 Qs) | ~4 hours | ~2 hours |
| EgoSchema subset (500 Qs) | ~40 min | ~20 min |

## 0. Setup

In [ ]:
# @title Clone the repository and install dependencies
import os

# ---------- CONFIGURE ----------
# Option A: Clone from GitHub (replace with your repo URL)
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # <-- EDIT THIS
BRANCH = "main"

# Option B: If you uploaded the code as a zip, set this path instead
# and comment out the git clone cell below.
# --------------------------------

PROJECT_ROOT = "/content/VAR"

if not os.path.exists(PROJECT_ROOT):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} /content/repo
    # Adjust if your repo root IS the VAR folder or contains it:
    !ln -sf /content/repo/VAR {PROJECT_ROOT} 2>/dev/null || \
     ln -sf /content/repo {PROJECT_ROOT}

# Install Python dependencies
!pip install -q -r {PROJECT_ROOT}/eval/requirements.txt rouge-score
!pip install -q -r {PROJECT_ROOT}/demo/backend/requirements.txt

print(f"\nProject root: {PROJECT_ROOT}")
print(f"Contents: {os.listdir(PROJECT_ROOT)}")

In [ ]:
# @title Alternative: Upload code as zip
# Uncomment this cell if you prefer uploading a zip of the VAR folder
# instead of cloning from GitHub.

# from google.colab import files
# uploaded = files.upload()  # Upload VAR.zip
# !unzip -q VAR.zip -d /content/
# PROJECT_ROOT = "/content/VAR"

In [ ]:
# @title Verify GPU and set up paths
import torch
import sys

assert torch.cuda.is_available(), (
    "No GPU detected! Go to Runtime → Change runtime type → GPU."
)
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

# Add backend to path so eval scripts can import StreamMind modules
BACKEND_DIR = f"{PROJECT_ROOT}/demo/backend"
EVAL_DIR = f"{PROJECT_ROOT}/eval"
RESULTS_DIR = f"{PROJECT_ROOT}/eval/results"

for p in [BACKEND_DIR, EVAL_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ["STREAMMIND_PROJECT_ROOT"] = PROJECT_ROOT

print(f"\nBackend: {BACKEND_DIR}")
print(f"Results: {RESULTS_DIR}")

In [ ]:
# @title Pre-download models (prevents timeout during evaluation)
print("Downloading models (first run only, ~2 GB)...")

from transformers import (
    CLIPModel, CLIPProcessor,
    BlipForConditionalGeneration, BlipProcessor,
    BlipForQuestionAnswering,
    T5ForConditionalGeneration, T5Tokenizer,
)

models = [
    ("CLIP ViT-B/32", lambda: (
        CLIPModel.from_pretrained("openai/clip-vit-base-patch32"),
        CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32"),
    )),
    ("BLIP Captioning", lambda: (
        BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base"),
        BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base"),
    )),
    ("BLIP VQA", lambda: (
        BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base"),
        BlipProcessor.from_pretrained("Salesforce/blip-vqa-base"),
    )),
    ("Flan-T5-base", lambda: (
        T5ForConditionalGeneration.from_pretrained("google/flan-t5-base"),
        T5Tokenizer.from_pretrained("google/flan-t5-base"),
    )),
]

for name, loader in models:
    print(f"  {name}...", end=" ", flush=True)
    loader()
    print("done")

print("\nAll models cached.")

## 1. Latency Profiling (GPU)

Measures per-component latency on GPU. These numbers replace the CPU estimates in the paper's latency table.

In [ ]:
# @title Download sample videos (required for evaluation)
import subprocess, shutil
from pathlib import Path

# Run the bundled download script
subprocess.run(
    [sys.executable, f"{PROJECT_ROOT}/demo/scripts/download_samples.py"],
    check=True,
)

# Create sample.mp4 symlink (run_docker_eval also looks for it)
sample_mp4 = Path(PROJECT_ROOT) / "demo" / "frontend" / "sample.mp4"
cooking_mp4 = Path(PROJECT_ROOT) / "demo" / "frontend" / "samples" / "cooking.mp4"
if not sample_mp4.exists() and cooking_mp4.exists():
    shutil.copy2(cooking_mp4, sample_mp4)
    print(f"Created sample.mp4 (copy of cooking.mp4)")

# Verify
video_files = list((Path(PROJECT_ROOT) / "demo" / "frontend").rglob("*.mp4"))
print(f"\nFound {len(video_files)} sample video(s):")
for v in video_files:
    size_mb = v.stat().st_size / (1024 * 1024)
    rel = str(v.relative_to(PROJECT_ROOT))
    print(f'  {rel:50s} {size_mb:.1f} MB')

In [ ]:
# @title Run latency profiling
import importlib, run_docker_eval as rde
importlib.reload(rde)
from run_docker_eval import profile_latency, _resolve_paths

# Re-resolve paths for Colab
rde.RESULTS_DIR, rde.VIDEOS = _resolve_paths(PROJECT_ROOT)
print(f"Videos available: {list(rde.VIDEOS.keys())}")
print(f"Results dir: {rde.RESULTS_DIR}")

latency = profile_latency(n_runs=10)

print("\n" + "=" * 55)
print("  GPU Latency Results")
print("=" * 55)
for comp, v in latency.items():
    print(f"  {comp:22s}  {v['mean_ms']:8.1f} ms  (std {v['std_ms']:.1f})")
print("=" * 55)

In [ ]:
# @title Format latency for paper table
import json

with open(f"{RESULTS_DIR}/latency.json") as f:
    lat = json.load(f)

print("% Paste into paper/sec/4_experiments.tex — Tab:latency")
print("% GPU column (measured on", gpu_name, ")")

mapping = [
    ("CLIP encoding (per frame)", "clip_encode"),
    ("SKM scoring + update", "skm_update"),
    ("TQR scope classification", "tqr_classify"),
    ("BLIP captioning (per frame)", "blip_caption"),
    ("BLIP VQA (per frame)", "blip_vqa"),
    ("Flan-T5 synthesis", "flan_t5"),
]

print()
for label, key in mapping:
    if key in lat:
        ms = lat[key]["mean_ms"]
        if ms < 1:
            val = "$<$1"
        else:
            val = f"{ms:.0f}" if ms >= 10 else f"$\\sim${ms:.0f}"
        print(f"    {label:35s} & {val} \\\\")
    else:
        print(f"    {label:35s} & TBD \\\\")

## 2. LiveQA-Bench Evaluation

Builds the LiveQA-Bench from sample videos and evaluates StreamMind (full model).

In [ ]:
# @title Build LiveQA-Bench and evaluate full model
import importlib, run_docker_eval as rde
importlib.reload(rde)
from run_docker_eval import build_liveqa_bench, evaluate_liveqa
from vlm_engine import VLMEngine

vlm = VLMEngine()
# force_rebuild=True to regenerate with fixed ground truth
qa_list = build_liveqa_bench(vlm, force_rebuild=True)
del vlm

print(f"\nBuilt {len(qa_list)} QA pairs")
from collections import Counter
scope_counts = Counter(q.scope for q in qa_list)
print(f"Scope distribution: {dict(scope_counts)}")

# Run full model evaluation
full_results = evaluate_liveqa(qa_list, memory_capacity=64, label="full")

In [ ]:
# @title Display LiveQA-Bench results
print("=" * 55)
print("  LiveQA-Bench — Full Model (N=64)")
print("=" * 55)
print(f"  Overall accuracy: {full_results['overall_accuracy']:.1f}%")
print(f"  Average score:    {full_results['avg_score']:.3f}")
print(f"  Average latency:  {full_results['avg_latency_ms']:.0f} ms")
print()
print("  Per-scope accuracy:")
for scope, acc in full_results["per_scope_accuracy"].items():
    n = full_results["scope_counts"][scope]
    print(f"    {scope:15s}  {acc:5.1f}%  (n={n})")
print("=" * 55)

# LaTeX line for paper
sa = full_results["per_scope_accuracy"]
oa = full_results["overall_accuracy"]
print("\n% LiveQA table row (Tab:liveqa):")
print(f"    \\textbf{{\\ours}} & \\textbf{{{sa.get('instant', 'TBD')}}} "
      f"& \\textbf{{{sa.get('recent', 'TBD')}}} "
      f"& \\textbf{{{sa.get('historical', 'TBD')}}} "
      f"& \\textbf{{{oa}}} \\\\")

## 3. Ablation Study

Tests different configurations on LiveQA-Bench: memory sizes (N=16, 32, 128), FIFO buffer, no TQR.

In [ ]:
# @title Run ablation study
from run_docker_eval import run_ablations

# Delete old saved benchmark so future runs use the corrected version
ablation_results = run_ablations(qa_list)

In [ ]:
# @title Display ablation results + LaTeX
print("=" * 60)
print("  Ablation Study Results")
print("=" * 60)

for name, res in ablation_results.items():
    print(f"  {name:12s}: acc={res['overall_accuracy']:5.1f}%  "
          f"score={res['avg_score']:.3f}  "
          f"scope_acc={res['per_scope_accuracy']}")

print("\n% Ablation table (Tab:ablation):")

label_map = {
    "full":   r"\textbf{\ours (full model)}",
    "fifo":   r"\quad w/o SKM (FIFO buffer, $N\!=\!64$)",
    "no_tqr": r"\quad w/o TQR (attend to full memory always)",
    "N16":    r"\quad $N = 16$ memory slots",
    "N32":    r"\quad $N = 32$ memory slots",
    "N128":   r"\quad $N = 128$ memory slots",
}

for key in ["full", "fifo", "no_tqr", "N16", "N32", "N128"]:
    if key in ablation_results:
        acc = ablation_results[key]["overall_accuracy"]
        label = label_map.get(key, key)
        bold = r"\textbf{" + f"{acc}" + "}" if key == "full" else str(acc)
        print(f"    {label:55s} & {bold} \\\\")

## 4. External Benchmarks (Optional)

To evaluate on NExT-QA, EgoSchema, OVO-Bench, and Ego4D-NLQ, you need to **upload the benchmark data** to Colab.

### Data Upload Options

**Option A — Google Drive** (recommended for large datasets):
1. Upload your benchmark data to Google Drive
2. Mount Drive and set the paths below

**Option B — Direct upload**:
1. Zip your data directories
2. Upload via the file browser or `files.upload()`

See `eval/README.md` for the expected directory layout per benchmark.

In [ ]:
# @title Mount Google Drive (run if your data is on Drive)
from google.colab import drive
drive.mount("/content/drive")

# Set your data paths here:
DRIVE_DATA = "/content/drive/MyDrive/streammind_data"  # <-- EDIT THIS

DATA_CONFIG = {
    "nextqa":    f"{DRIVE_DATA}/nextqa",
    "egoschema": f"{DRIVE_DATA}/egoschema",
    "ovobench":  f"{DRIVE_DATA}/ovobench",
    "ego4d_nlq": f"{DRIVE_DATA}/ego4d",
    "liveqa":    f"{DRIVE_DATA}/liveqa",
}

# Verify which datasets are available
for name, path in DATA_CONFIG.items():
    exists = os.path.isdir(path)
    status = "found" if exists else "NOT FOUND"
    print(f"  {name:15s} {status:12s} {path}")

In [ ]:
# @title Run external benchmark evaluation
# This cell evaluates on whichever benchmarks have data available.
# Each benchmark takes 20 min to several hours on GPU.

from evaluate import evaluate_benchmark
from pipeline import EvalPipeline

pipeline = EvalPipeline(
    memory_capacity=64,
    sample_fps=2.0,
    frame_skip=1,
    device="cuda",
)

ext_results = {}

# Choose which benchmarks to run (comment out ones you want to skip)
BENCHMARKS_TO_RUN = [
    # "nextqa",      # ~2-4 hours on GPU
    # "egoschema",   # ~20-40 min on GPU
    # "ovobench",    # ~1-2 hours on GPU
    # "ego4d_nlq",   # ~30 min on GPU
]

for bench_name in BENCHMARKS_TO_RUN:
    data_root = DATA_CONFIG.get(bench_name)
    if not data_root or not os.path.isdir(data_root):
        print(f"Skipping {bench_name}: data not found at {data_root}")
        continue

    print(f"\n{'='*60}")
    print(f"  Evaluating: {bench_name}")
    print(f"{'='*60}")

    kwargs = {}
    if bench_name == "egoschema":
        kwargs["subset"] = "subset"

    try:
        summary = evaluate_benchmark(
            bench_name, data_root, pipeline,
            max_samples=0,
            compute_gpt_score=False,
            output_dir=RESULTS_DIR,
            **kwargs,
        )
        ext_results[bench_name] = summary
        print(f"\n{bench_name} accuracy: {summary['accuracy']}%")
    except Exception as e:
        print(f"Error on {bench_name}: {e}")

if ext_results:
    with open(f"{RESULTS_DIR}/external_summaries.json", "w") as f:
        json.dump(ext_results, f, indent=2)
    print(f"\nExternal results saved to {RESULTS_DIR}/external_summaries.json")

## 5. Generate LaTeX Values for the Paper

Reads all result JSON files and prints the exact values to replace `\tbd` placeholders in `paper/sec/4_experiments.tex`.

In [ ]:
# @title Generate all LaTeX values
import json
from pathlib import Path

results_path = Path(RESULTS_DIR)

print("=" * 60)
print("  LaTeX Values for paper/sec/4_experiments.tex")
print("=" * 60)

# --- LiveQA-Bench (from run_docker_eval output) ---
liveqa_path = results_path / "liveqa_full.json"
if liveqa_path.exists():
    with open(liveqa_path) as f:
        liveqa = json.load(f)["summary"]
    sa = liveqa["per_scope_accuracy" if "per_scope_accuracy" in liveqa else "per_scope"]
    oa = liveqa["overall_accuracy"]
    print(f"\n% --- LiveQA-Bench (Tab:liveqa) ---")
    print(f"    \\textbf{{\\ours}} & \\textbf{{{sa.get('instant', 'TBD')}}} "
          f"& \\textbf{{{sa.get('recent', 'TBD')}}} "
          f"& \\textbf{{{sa.get('historical', 'TBD')}}} "
          f"& \\textbf{{{oa}}} \\\\")

# --- Ablation (from run_docker_eval output) ---
ablation_path = results_path / "ablation_summary.json"
if ablation_path.exists():
    with open(ablation_path) as f:
        abl = json.load(f)
    print(f"\n% --- Ablation (Tab:ablation) ---")
    for key in ["full", "fifo", "no_tqr", "N16", "N32", "N128"]:
        if key in abl:
            print(f"%   {key:10s} -> {abl[key]['overall_accuracy']}%")

# --- Latency ---
lat_path = results_path / "latency.json"
if lat_path.exists():
    with open(lat_path) as f:
        lat = json.load(f)
    print(f"\n% --- Latency (Tab:latency, GPU column) ---")
    for comp, v in lat.items():
        print(f"%   {comp:22s} = {v['mean_ms']:.1f} ms")

# --- External benchmarks (from evaluate.py output) ---
for bench in ["nextqa", "egoschema", "ovobench", "ego4d_nlq"]:
    sp = results_path / f"{bench}_summary.json"
    if sp.exists():
        with open(sp) as f:
            s = json.load(f)
        print(f"\n% --- {bench} ---")
        print(f"%   Accuracy: {s['accuracy']}%")
        if "gpt_score" in s:
            print(f"%   GPT Score: {s['gpt_score']}")
        if "per_category" in s:
            for k, v in s["per_category"].items():
                print(f"%     {k}: {v}%")
        if "recall_at_1_iou03" in s:
            print(f"%   R@1 IoU>=0.3: {s['recall_at_1_iou03']}%")

print("\n" + "=" * 60)
print("  Copy the above values into paper/sec/4_experiments.tex")
print("  Replace \\tbd placeholders with actual numbers.")
print("=" * 60)

In [ ]:
# @title Run the dedicated results_to_latex.py script
!cd {EVAL_DIR} && python results_to_latex.py --results-dir {RESULTS_DIR}

## 6. Download Results

Download all result JSONs so you can use them locally.

In [ ]:
# @title List all result files
result_files = sorted(Path(RESULTS_DIR).glob("*.json"))
print(f"Result files ({len(result_files)}):")
for f in result_files:
    size_kb = f.stat().st_size / 1024
    print(f"  {f.name:40s} {size_kb:6.1f} KB")

In [ ]:
# @title Download results as zip
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/streammind_results", "zip", RESULTS_DIR)
print(f"Created: {zip_path}")
files.download(zip_path)

In [ ]:
# @title (Optional) Save results to Google Drive
import shutil

DRIVE_SAVE = "/content/drive/MyDrive/streammind_results"
os.makedirs(DRIVE_SAVE, exist_ok=True)

for f in Path(RESULTS_DIR).glob("*.json"):
    shutil.copy2(f, DRIVE_SAVE)
    print(f"  Copied {f.name}")

print(f"\nResults saved to {DRIVE_SAVE}")

## 7. Updating the Paper

After running the evaluation, follow these steps to add results to the paper:

### Step 1: Copy result JSONs to your local machine
Download the zip from Section 6 above and extract into `VAR/eval/results/`.

### Step 2: Generate LaTeX values
```bash
cd VAR/eval
python results_to_latex.py --results-dir ./results
```

### Step 3: Replace `\tbd` in the paper
Open `paper/sec/4_experiments.tex` and replace each `\tbd` with the corresponding number from the script output.

### Step 4: Update latency table
In the latency table (`Tab:latency`), replace the GPU estimates with the actual measured values from Phase 1.

### Step 5: Update the ablation table
Fill in the remaining ablation rows (`Tab:ablation`) with the values from Phase 3.